In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql import Window

spark = SparkSession \
    .builder \
    .master("local") \
    .config("spark.driver.memory", "4g") \
    .appName("ex3_anomalies_detection") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/06 19:22:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/06 19:22:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
sliding_range_window = Window.partitionBy(F.col("carrier")).orderBy(F.col("start_range"))

In [3]:
flights_df = spark.read.parquet('s3a://pyspark/data/trasformed/flights/', header=True)

flights_df.cache()

26/08/06 19:23:48 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[day_of_week: int, day_of_month: int, carrier: string, origin_airport_id: int, dest_airport_id: int, dep_delay: int, arr_delay: int, flight_date: date]

In [10]:
# Group Data with Sliding Window: The data is grouped by Carrier and a sliding window of 10 days with a 1-day step. 
# The resulting total delay (sum of departure and arrival delays) for each window is calculated.

grouped_df = flights_df \
.groupBy(F.col('carrier'),F.window(F.col('flight_date'),'10 days','1 day').alias('date window')) \
.agg(F.sum(F.col('arr_delay') + F.col('dep_delay')).alias("total_delay"))

In [11]:
# Select Relevant Columns: The results are structured to contain the Carrier, 
# the start and end of the window, and the computed total_delay.

structured_df = grouped_df\
.select(
    F.col("carrier"),
    F.col("date window.start").alias("start_range"),
    F.col("date window.end").alias("end_range"),
    F.col("total_delay")
)

In [12]:
# Calculate Percentage Change: For each row, find the delay from the previous window and calculate the percentage change.

change_df = structured_df.withColumn("last_window_delay", F.lag(F.col("total_delay")).over(sliding_range_window)) \
.withColumn("change_percent", F.abs(F.lit(1.0) - (F.col("total_delay") / F.col("last_window_delay"))))

In [13]:
significant_changes_df = change_df.where(F.col('change_percent') > F.lit(0.3))
significant_changes_df.show(100)

+-------+-------------------+-------------------+-----------+-----------------+-------------------+
|carrier|        start_range|          end_range|total_delay|last_window_delay|     change_percent|
+-------+-------------------+-------------------+-----------+-----------------+-------------------+
|     9E|2020-04-23 00:00:00|2020-05-03 00:00:00|       8931|              797|  10.20577164366374|
|     9E|2020-04-24 00:00:00|2020-05-04 00:00:00|      19453|             8931|  1.178143544955772|
|     9E|2020-04-28 00:00:00|2020-05-08 00:00:00|      32581|            19974| 0.6311705216781816|
|     9E|2020-05-01 00:00:00|2020-05-11 00:00:00|      53707|            35586| 0.5092171078514023|
|     9E|2020-05-04 00:00:00|2020-05-14 00:00:00|      76713|            57455| 0.3351840570881559|
|     9E|2020-05-29 00:00:00|2020-06-08 00:00:00|      47306|            71614|0.33943083754573133|
|     9E|2020-05-30 00:00:00|2020-06-09 00:00:00|      27224|            47306|0.42451274679744644|
